In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "cloudtrail.db"

assert db_path.exists(), "SQLite database not found"

connection = sqlite3.connect(db_path)

In [2]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_events FROM events",
    connection
)

,total_events
0,95


In [24]:
# Started excercise at 7:00am
pd.read_sql_query(
    """
    WITH failed_events AS(
        SELECT
            user,
            timestamp AS most_recent_failure,
            RANK() OVER (ORDER BY timestamp DESC) AS rank
        FROM events
        WHERE failed = 1
        GROUP BY user
    )
        SELECT
            e.user,
            COUNT(*) as total_events,
            fe.most_recent_failure
        FROM events as e
        JOIN failed_events as fe
            ON e.user = fe.user
    """, connection
)

,user,total_events,most_recent_failure
0,admin,77,2026-11-11T07:00:00Z


In [ ]:
# Started excercise at 7:30am
pd.read_sql_query(
    """
    SELECT
            user,
            timestamp AS most_recent_failure,
            RANK() OVER (ORDER BY timestamp DESC) AS rank
        FROM events
        WHERE failed = 1
        GROUP BY user
""",
connection
)

,user,most_recent_failure,rank
0,developer,2026-11-14T01:00:00Z,1
1,admin,2026-11-11T07:00:00Z,2
2,service-user,2026-11-11T04:00:00Z,3


In [67]:
df_day20 = pd.read_sql_query(
    """
    WITH recent_failure AS (
        SELECT
            user,
            MAX(timestamp) AS most_recent_failure
        FROM events
        WHERE failed = 1
        GROUP BY user
    ),
    total_events_tb AS (
        SELECT
            user,
            COUNT(*) as total_events
        FROM events
        GROUP BY user
    )
    SELECT   
        te.user,
        te.total_events,
        rf.most_recent_failure
    FROM total_events_tb as te
    LEFT JOIN recent_failure as rf
        ON te.user=rf.user
    ORDER BY te.total_events DESC
    """, connection
    )
df_day20

,user,total_events,most_recent_failure
0,admin,36,2026-11-14T05:00:00Z
1,developer,22,2026-11-14T22:00:00Z
2,service-user,19,2026-11-12T17:00:00Z
3,analyst,18,NaN


In [62]:
assert len(df_day20) == 4
assert df_day20["total_events"].sum() == 95

assert df_day20.loc[
    df_day20["user"] == "analyst",
    "most_recent_failure"
].isna().all()

In [75]:
df_day20_2 = pd.read_sql_query(
    """
    WITH ranked_failures AS (
        SELECT
            user,
            timestamp,
            action,
            ROW_NUMBER() OVER (
                PARTITION BY user
                ORDER BY timestamp DESC
            ) AS row_num
        FROM events
        WHERE failed = 1
),
    total_events_tb AS (
        SELECT
            user,
            COUNT(*) as total_events
        FROM events
        GROUP BY user
    )
    SELECT   
        te.user,
        te.total_events,
        rf.timestamp AS most_recent_failure,
        rf.action
    FROM total_events_tb as te
    LEFT JOIN ranked_failures as rf
        ON te.user=rf.user
        AND rf.row_num = 1
    ORDER BY te.total_events DESC
    """, connection
    )
df_day20_2

,user,total_events,most_recent_failure,action
0,admin,36,2026-11-14T05:00:00Z,PutObject
1,developer,22,2026-11-14T22:00:00Z,ConsoleLogin
2,service-user,19,2026-11-12T17:00:00Z,PutObject
3,analyst,18,NaN,NaN


In [76]:
assert len(df_day20_2) == 4
assert df_day20_2["total_events"].sum() == 95

assert df_day20_2.loc[
    df_day20_2["user"] == "analyst",
    "most_recent_failure"
].isna().all()

assert df_day20_2.loc[
    df_day20_2["user"] == "analyst",
    "action"
].isna().all()

### Day 20 — Interview Error Log

**Problem:** Find each identity's total events and most recent failed event, including the associated action.

**Initial mistakes:**
- Used incorrect syntax when declaring multiple CTEs.
- Attempted to combine `GROUP BY` and `RANK()` without correctly identifying the latest failure per user.
- Initially omitted the required ordering.
- Later used `WHERE row_num = 1` after a `LEFT JOIN`, unintentionally excluding identities without failures.

**Corrections:**
- Declared multiple CTEs using one `WITH` clause.
- Used `MAX(timestamp)` for the simpler latest-timestamp requirement.
- Used `ROW_NUMBER()` with `PARTITION BY user` to retrieve attributes from the latest failed event.
- Moved `rf.row_num = 1` into the `LEFT JOIN` condition to preserve unmatched identities.

**Key lesson:** Filtering a right-hand table in the `WHERE` clause after a `LEFT JOIN` can remove unmatched rows. When unmatched identities must remain, filter the matching records in the `ON` clause or in a separate CTE.

**Interview takeaway:** Always validate row counts, confirm how `NULL` values are handled, and check that the query satisfies every requirement—not just that it executes successfully.

### Day 20 — Independent SQL Interview Practice

Completed an independent SQL interview exercise using the Cloud Security Intelligence Platform's 95 deduplicated synthetic CloudTrail events.

**Objective:** Retrieve each identity's total event count and most recent failed event while preserving identities without failures.

**Key accomplishments:**
- Completed an initial SQL attempt without step-by-step assistance.
- Used multiple CTEs to organize SQL aggregations.
- Calculated total event counts using `COUNT(*)` and `GROUP BY`.
- Retrieved the most recent failed timestamp using `MAX(timestamp)`.
- Extended the solution using `ROW_NUMBER()` and `PARTITION BY` to retrieve the action associated with each identity's latest failure.
- Corrected a `LEFT JOIN` filtering mistake by moving `rf.row_num = 1` from the `WHERE` clause into the `ON` clause.
- Validated the final results with pandas assertions.

**Results:**
- Returned four identities accounting for all 95 events.
- Correctly preserved the analyst identity despite having no failed events.
- Confirmed that missing failure timestamps and actions are represented by null values.

**Key lessons:**
1. `MAX(timestamp)` is often sufficient when only the most recent timestamp is required.
2. `ROW_NUMBER()` with `PARTITION BY` is useful when retrieving additional attributes from the latest event.
3. Filtering a right-side table using `WHERE` after a `LEFT JOIN` can unintentionally eliminate unmatched records.
4. SQL queries should be validated against business requirements, expected row counts, and edge cases.

**Limitations:** The dataset is synthetic. The final ranking query does not have a deterministic tie-breaker when multiple failed events share an identical timestamp.

**Reflection:** My initial solution contained syntax and logic mistakes, but I corrected them through debugging and validation. I learned to pay closer attention to SQL execution behavior, join filtering, and preserving records with missing values.